In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain.agents import create_agent

In [7]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_community.vectorstores import InMemoryVectorStore




In [9]:
pdfloader = PyPDFLoader("../data/rag-pdf3.pdf")

pdfs = pdfloader.load()

In [10]:
splitter=RecursiveCharacterTextSplitter(chunk_size=200,chunk_overlap=50)
splitted_data=splitter.split_documents(pdfs)

In [11]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/Users/sawals/Documents/GitHub/GENAI_SERIES_PRACTICE/agentic_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1101 20:39:22.906000 56933 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8036.86it/s]


In [12]:
vector_store= Chroma.from_documents(
    documents=splitted_data,
    embedding=embeddings,
    persist_directory="./vector_db"
)

In [22]:
def retriever_tool(query: str):
    """
        This tool can help you to retrieve the relevant data of the PDF documents and these documents have details about patient medical details.
    """
    docs=vector_store.similarity_search(query=query)
    context=""
    for doc in splitted_data:
        context+=doc.page_content +"\n"
    
    return context

In [14]:
llm = ChatGroq(model="openai/gpt-oss-20b",streaming="True")

In [23]:
System_Prompt="""
    You are a helpful assistant that answers questions using retrieved context.
    ALWAYS use the 'retriever_tool' tool for questions requiring external knowledge.
"""

In [24]:
agent = create_agent(
    model=llm,
    tools=[retriever_tool],
    system_prompt=System_Prompt
)

In [25]:
query="What is the name of patient and what is his heartbeat?"
response= agent.invoke({"messages":[{"role":"user", "content":query}]})

print(response["messages"][-1].content)

**Patient name:** Robert Vance Miller  
**Heartbeat (heart rate):** 92 beats per minute (sinus rhythm)
